# Lesson 07 - ಯೋಜನೆ ವಿನ್ಯಾಸ ಮಾದರಿ

ಈ ನೋಟ್ಬುಕ್ ಮೈಕ್ರೋಸಾಫ್ಟ್ ಏಜೆಂಟ್ ಫ್ರೇಮ್ವರ್ಕ್ ಬಳಸಿ AI ಏಜೆಂಟುಗಳಿಗಾಗಿ **ಯೋಜನೆ ವಿನ್ಯಾಸ ಮಾದರಿ** ಅನ್ನು ವಿವರಿಸುತ್ತದೆ.
ನೀವು ಸಂಕೀರ್ಣ ಪ್ರಯಾಣ ವಿನಂತಿಗಳನ್ನು ರಚನೆಗೊಳಿಸಿದ ಉಪಕಾರ್ಯಗಳಾಗಿ ಹೇಗೆ ವಿಭಜಿಸಬೇಕು, ಅವುಗಳನ್ನು ತಜ್ಞ ಏಜೆಂಟ್‌ಗಳಿಗೆ ಹೇಗೆ ಹಂಚಿಕೊಡಬೇಕು,
ಮತ್ತು ಫಲಿತಾಂಶದ ಯೋಜನೆಯನ್ನು ಹೇಗೆ ಕಾರ್ಯಗತಗೊಳಿಸಬೇಕು ಎಂದು ಕಲಿಯೋದು — ಇವುಗಳನ್ನು Pydantic ಮಾದರಿಗಳಿಂದ ಶಕ್ತಿ ಪಡೆದ ರಚಿತ output ಬಳಸಿ ಮಾಡಲಾಗುತ್ತದೆ.


## ಸೆಟಪ್


In [ ]:
! pip install agent-framework azure-ai-projects azure-identity -U -q

In [ ]:
import logging
logging.getLogger("agent_framework.azure").setLevel(logging.ERROR)

import os, asyncio
from typing import Annotated
from pydantic import BaseModel
from agent_framework import tool
from agent_framework.azure import AzureAIProjectAgentProvider
from azure.identity import AzureCliCredential

In [ ]:

provider = AzureAIProjectAgentProvider(credential=AzureCliCredential())

## ಕಾರ್ಯ ವಿಭಜನೆ

ಕಾರ್ಯ ವಿಭಜನೆ ಯೋಜನೆ ವಿನ್ಯಾಸ ಮಾದರಿಯ ಮುಖಾಂತರವಾಗಿರುವ ಕೇಂದ್ರ ಭಾಗವಾಗಿದೆ. ಒಂದು ಸಂಕೀರ್ಣ ವಿನಂತಿಯನ್ನು ಆರಂಭದಿಂದ ಅಂತ್ಯಕ್ಕೆ ಒಬ್ಬ ಏಜಂಟಿಗೆ ನಿಭಾಯಿಸಲು ಬದಲು, ನಾವು ಸಮಸ್ಯೆಯನ್ನು ಚಿಕ್ಕ, ಚೆನ್ನಾಗಿ ವ್ಯಾಖ್ಯಾನಿಸಲಾದ **ಉಪಕಾರ್ಯಗಳು** ಆಗಿ ವಿಭಜಿಸುತ್ತೇವೆ. ಪ್ರತಿ ಉಪಕಾರ್ಯವನ್ನು ಸ್ಪಷ್ಠ ಆದ್ಯತೆಗಳು ಮತ್ತು ಅವಲಂಬನೆ ಕ್ರಮಗಳೊಂದಿಗೆ ವಿಶಿಷ್ಟ ಏಜಂಟಿಗೆ (ಉದಾಹರಣೆಗೆ, ವಿಮಾನಗಳು, ಹೋಟೆಲ್ಗಳು, ಚಟುವಟಿಕೆಗಳು) ಆಯೋಜಿಸಲಾಗುತ್ತದೆ.

ಇದರಿಂದ ಹಲವಾರು ಲಾಭಗಳು ಲಭ್ಯವಾಗುತ್ತವೆ:
- **ಸ್ಪಷ್ಟತೆ**: ಪ್ರತಿ ಉಪಕಾರ್ಯಕ್ಕೆ ಒಂದಿಷ್ಟು ಜವಾಬ್ದಾರಿ ಇರುತ್ತದೆ.
- **ಸಮಂತರ ಚಾಲನೆ**: ಸ್ವಾತಂತ್ರ್ಯ ಉಪಕಾರ್ಯಗಳು ಸಮಕಾಲೀನವಾಗಿ ನಡೆಯಬಹುದು.
- **ನಂಬಕಸ್ಥಿತಿ**: ವೈಫಲ್ಯಗಳು ವೈಯಕ್ತಿಕ ಉಪಕಾರ್ಯಗಳಲ್ಲಿ ಮಾತ್ರ ಇರುವವು.
- **ಬಜೆಟ್ ಟ್ರ್ಯಾಕಿಂಗ್**: ವೆಚ್ಚಗಳು ಪ್ರತಿ ಉಪಕಾರ್ಯಕ್ಕಾಗಿಯೇ ಅಂದಾಜು ಮಾಡಲಾಗುತ್ತವೆ ಮತ್ತು ಸಂಗ್ರಹವಾಗುತ್ತವೆ.


In [ ]:
class TravelSubTask(BaseModel):
    task_id: int
    description: str
    assigned_agent: str  # "flight_agent", "hotel_agent", "activity_agent"
    priority: str  # "high", "medium", "low"
    dependencies: list[int] = []


class TravelPlan(BaseModel):
    destination: str
    trip_duration_days: int
    subtasks: list[TravelSubTask]
    total_estimated_budget_usd: int
    notes: str

## ರಚಿಸಲಾದ ಔಟ್‌ಪುಟ್ನೊಂದಿಗೆ ಯೋಜನಾ ಏಜೆಂಟ್ ರಚನೆ

ಯೋಜನಾ ಏಜೆಂಟ್ **ಮುಂಚೂಣಿ ಸಹಾಯಕ** ಆಗಿ ಕಾರ್ಯನಿರ್ವಹಿಸುತ್ತದೆ. ಒಂದು ಉನ್ನತ ಮಟ್ಟದ ಪ್ರಯಾಣ ವಿನಂತಿ ನೀಡಿದಾಗ, ಅದು ರಚಿಸಲಾದ `TravelPlan` ಅನ್ನು ಉತ್ಪಾದಿಸುತ್ತದೆ — ವಿನಂತಿಯನ್ನು ಉಪಕಾರ್ಯಗಳಾಗಿ ವಿಭಜಿಸಿ,.prioritiesನಿರ್ದೇಶಿಸಿ, ಮತ್ತು ಆತಥಿತತೆಗಳನ್ನು ಗುರುತಿಸಿ, ಇದರಿಂದ ಒಂದುconcierge ಅಥವಾ ಕಾರ್ಯ ನಿರ್ವಹಣಾ ಪದರ ಕಾರ್ಯವನ್ನು ನಿರ್ವಹಿಸಬಹುದು.


In [ ]:
planning_agent = await provider.create_agent(
    name="TravelPlanner",
    instructions="""You are a travel planning agent. When given a travel request:
1. Break it into specific subtasks (flights, hotels, activities, logistics)
2. Assign each subtask to the appropriate specialist agent
3. Set priorities and identify dependencies between tasks
4. Estimate the total budget""",
)

result = await planning_agent.run(
    "Plan a 7-day trip to Paris for a couple interested in art, cuisine, and history. Budget around $5000.",
    )
if result:
    plan = result
    print(f"Destination: {plan.destination}")
    print(f"Duration: {plan.trip_duration_days} days")
    print(f"Budget: ${plan.total_estimated_budget_usd}")
    print(f"\nSubtasks:")
    for task in plan.subtasks:
        print(f"  [{task.priority}] {task.task_id}. {task.description} → {task.assigned_agent}")

## ವಿಶಿಷ್ಠ ಸಾಧನಗಳೊಂದಿಗೆ ಯೋಜನೆಯನ್ನು ಜರುಗಿಸುವುದು

ಮುಂದಿನ ಡೆಸ್ಕ್ ಏಜೆಂಟ್ ರಚಿಸಲಾದ ಸ್ಥಿರವಾದ ಯೋಜನೆಯನ್ನು, **ಕನ್ಸಿಯರ್ಜ್ ಏಜೆಂಟ್** ಜರುಗಿಸುತ್ತಾರೆ. ಪ್ರತಿಯೊಂದು ವಿಶಿಷ್ಟ ಸಾಧನವು ಒಂದು ಉಪಕಾರ್ಯದ ವರ್ಗವನ್ನು (ವಿಮಾನಗಳು, ಹೊಟೇಲುಗಳು, ಚಟುವಟಿಕೆಗಳು) ನಿರ್ವಹಿಸುತ್ತದೆ. ಕನ್ಸಿಯರ್ಜ್ ಯೋಜನೆಯ ಉಪಕಾರ್ಯಗಳನ್ನು ಅವಲಂಬನೆ ಆಧಾರದ ಮೇಲೆ ಕ್ರಮವಾಗಿ ಸರಿದೂಗುಸಿ ಸೂಕ್ತ ಸಾಧನಕ್ಕೆ ಪ್ರತಿ ಒಂದನ್ನು ವಿತರಿಸುತ್ತದೆ.


In [ ]:
@tool
def book_flight(
    destination: Annotated[str, "The destination city"],
    departure_date: Annotated[str, "Departure date (YYYY-MM-DD)"],
    return_date: Annotated[str, "Return date (YYYY-MM-DD)"],
) -> str:
    """Search and book flights for the trip."""
    return f"Flight booked to {destination}: {departure_date} → {return_date}, confirmation #FLT-{hash(destination) % 10000:04d}"


@tool
def reserve_hotel(
    city: Annotated[str, "The city for the hotel"],
    check_in: Annotated[str, "Check-in date (YYYY-MM-DD)"],
    check_out: Annotated[str, "Check-out date (YYYY-MM-DD)"],
    guests: Annotated[int, "Number of guests"],
) -> str:
    """Reserve a hotel room in the destination city."""
    return f"Hotel reserved in {city}: {check_in} to {check_out} for {guests} guests, confirmation #HTL-{hash(city) % 10000:04d}"


@tool
def book_activity(
    activity_name: Annotated[str, "Name of the activity or tour"],
    date: Annotated[str, "Date of the activity (YYYY-MM-DD)"],
    participants: Annotated[int, "Number of participants"],
) -> str:
    """Book a tour, museum visit, or other activity."""
    return f"Activity booked: {activity_name} on {date} for {participants} people, confirmation #ACT-{hash(activity_name) % 10000:04d}"


# Concierge agent that executes the plan using specialist tools
concierge_agent = await provider.create_agent(
    name="Concierge",
    instructions="""You are a travel concierge executing a structured travel plan.
Use the available tools to fulfil each subtask. Work through the subtasks in order,
respecting dependencies. Summarise the results when finished.""",
    tools=[book_flight, reserve_hotel, book_activity],
)

# Build a prompt from the plan produced above
if result.value:
    subtask_lines = "\n".join(
        f"- [{t.priority}] {t.task_id}. {t.description} (agent: {t.assigned_agent}, deps: {t.dependencies})"
        for t in plan.subtasks
    )
    execution_prompt = (
        f"Execute the following travel plan for {plan.destination} "
        f"({plan.trip_duration_days} days, ${plan.total_estimated_budget_usd} budget):\n"
        f"{subtask_lines}"
    )

    exec_response = await concierge_agent.run(execution_prompt)
    print(exec_response)

## ಸಾರಾಂಶ

ಈ ಪಾಠದಲ್ಲಿ ನೀವು AI ಏಜೆಂಟ್‌ಗಳಿಗಾಗಿ **ಯೋಜನೆ ವಿನ್ಯಾಸ ಮಾದರಿ** ಅನ್ನು ಕಲಿತಿರಿ:

1. **ಕಾರ್ಯ ವಿಭಜನೆ** — ಮುಂದಣ ಡೆಸ್ಕ್ ಯೋಜನಾ ಏಜೆಂಟ್ ಸಂಕೀರ್ಣ ಪ್ರಯಾಣ ವಿನಂತಿಯನ್ನು Pydantic ಮಾದರಿಗಳನ್ನು ಬಳಸಿ ಕಟ್ಟಿಸಲಾಗಿರುವ ಉಪಕಾರ್ಯಗಳಲ್ಲಿ ವಿಭಜಿಸಿ, ಪ್ರತಿ ಉಪಕಾರ್ಯವನ್ನು ಪ್ರಾಥಮಿಕತೆ ಮತ್ತು ಅವಲಂಬನೆಗಳನ್ನು ಹೊಂದಿರುವ ವಿಶೇಷಜ್ಞ ಏಜೆಂಟ್‌ಗೆ ನೀಡುತ್ತದೆ.
2. **ಸಂರಚಿತ ಫಲಿತಾಂಶ** — `response_format` ಅನ್ನು ನೀಡಿ, ಏಜೆಂಟ್ ಮುಕ್ತರೂಪ ಪಠ್ಯವನ್ನು ಬದಲಿ ಸರಿಹೊಂದಿಸಿದ `TravelPlan` ವಸ್ತುವನ್ನು ಹಿಂದಿರುಗಿಸುತ್ತದೆ, ಇದು ನಂತರದ ಪ್ರಕ್ರಿಯೆಯನ್ನು ನಂಬಿಗಸ್ಥಗೊಳಿಸುತ್ತದೆ.
3. **ಯೋಜನೆ ನಿರ್ವಹಣೆ** — ಕಾನ್ಸರ್ಜ್ ಏಜೆಂಟ್ ವಿಶೇಷಜ್ಞ ಸಾಧನಗಳನ್ನು (`book_flight`, `reserve_hotel`, `book_activity`) ಉಪಯೋಗಿಸಿ ಉಪಕಾರ್ಯಗಳನ್ನು ಕ್ರಮವಾಗಿ ನಿರ್ವಹಿಸಿ, ಯೋಜನೆಯನ್ನು ಅನುಷ್ಠಾನಗೊಳಿಸಿ ಮತ್ತು ಫಲಿತಾಂಶಗಳನ್ನು ವರದಿ ಮಾಡುತ್ತದೆ.

ಈ ಮಾದರಿ *ಏನು ಮಾಡಬೇಕು* (ಯೋಜನೆ) ಮತ್ತು *ಏಕೆಗೆ/how* (ನಿರ್ವಹಣೆ) ಎಂಬುದನ್ನು ಬೇರ್ಪಡಿಸುವ ಮೂಲಕ ಏಜೆಂಟ್‌ಗಳನ್ನು ಹೆಚ್ಚು ಘಟಕಗೊಳಿಸುವುದು, ಪರೀಕ್ಷಿಸಲು ಸರಳಗೊಳಿಸುವುದು ಮತ್ತು ವಿಸ್ತರಿಸಲು ಸುಲಭವನ್ನಾಗಿಸುತ್ತದೆ.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**ತಪ್ಪು ಹೇಳಿಕೆ**:  
ಈ ದಸ್ತಾವೇಜು AI ಭಾಷಾಂತರ ಸೇವೆಯಾದ [Co-op Translator](https://github.com/Azure/co-op-translator) ಬಳಸಿ ಭಾಷಾಂತರಿಸಲಾಗಿದೆ. ನಾವು ಶುದ್ಧತೆಯತ್ತ ಪ್ರಯತ್ನಿಸುತ್ತಿದ್ದರೂ, ಸ್ವಯಂಚಾಲಿತ ಭಾಷಾಂತರಗಳಲ್ಲಿ ದೋಷಗಳು ಅಥವಾ ಅನಿಶ್ಚಿತತೆಗಳಿರಬಹುದು ಎಂದು ದಯವಿಟ್ಟು గమనಿಸಿ. ಮೂಲ ದಸ್ತಾವೇಜನ್ನು ಅದರ ಮೂಲ ಭಾಷೆಯಲ್ಲಿ ಅಧಿಕೃತ ಮೂಲ ಎಂದು ಪರಿಗಣಿಸುವುದು ಉತ್ತಮ. ಮಹತ್ವದ ಮಾಹಿತಿಗಾಗಿ, ವೃತ್ತಿಪರ ಮಾನವ ಭಾಷಾಂತರವನ್ನು ಶಿಫಾರಸು ಮಾಡಲಾಗುತ್ತದೆ. ಈ ಭಾಷಾಂತರ ಬಳಕೆಯಿಂದ ಉಂಟಾಗುವ ಯಾವುದೇ ಅರ್ಥ ವ್ಯತ್ಯಾಸಗಳು ಅಥವಾ ತಪ್ಪು ಅರ್ಥಮಾಡಿಕೊಳ್ಳಿಕೆಗೆ ನಾವು ಹೊಣೆಗಾರರಲ್ಲ.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
